## Example 1: Create STAC Catalog and Collection

In [ ]:
import pystac
from pystac import Catalog, Collection
import datetime

# Example 1: Create STAC Catalog and Collection
print("Testing pystac Catalog and Collection:")

# Create catalog
catalog = Catalog(id='test-catalog', description='Test catalog')
print(f"✓ Created catalog: {catalog.id}")
print(f"✓ STAC version: {pystac.get_stac_version()}")

# Create collection with extent
collection = Collection(
    id='test-collection',
    description='Test collection',
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),
        temporal=pystac.TemporalExtent([[datetime.datetime(2020, 1, 1), None]])
    )
)
print(f"✓ Created collection: {collection.id}")
print(f"✓ Spatial extent: {collection.extent.spatial.bboxes[0]}")
print(f"✓ Temporal start: {collection.extent.temporal.intervals[0][0]}")

# Add collection to catalog
catalog.add_child(collection)
print(f"✓ Added collection to catalog")
print(f"✓ Catalog children: {len(list(catalog.get_children()))}")
assert catalog.id == 'test-catalog' and collection.id == 'test-collection'
assert pystac.get_stac_version().startswith('1.')
assert [child.id for child in catalog.get_children()] == ['test-collection']
assert collection.get_parent() is catalog
assert collection.extent.spatial.bboxes[0] == [-180, -90, 180, 90]
assert collection.extent.temporal.intervals[0] == [datetime.datetime(2020, 1, 1), None]

## Example 2: Create STAC Item with Assets

In [ ]:
from pystac import Item, Asset
import datetime

# Example 2: Create STAC Item with Assets
print("Testing pystac Item and Assets:")

# Create item with geometry
item = Item(
    id='test-item-001',
    geometry={"type": "Point", "coordinates": [-122.4, 37.8]},
    bbox=[-122.4, 37.8, -122.4, 37.8],
    datetime=datetime.datetime(2023, 6, 15, 12, 0, 0),
    properties={'platform': 'satellite-1', 'cloud_cover': 15.5}
)

print(f"✓ Created item: {item.id}")
print(f"✓ Datetime: {item.datetime}")
print(f"✓ Properties: {item.properties}")

# Add assets
item.add_asset('visual', Asset(
    href='https://example.com/visual.tif',
    media_type='image/tiff; application=geotiff',
    roles=['visual']
))
item.add_asset('metadata', Asset(
    href='https://example.com/metadata.json',
    media_type='application/json',
    roles=['metadata']
))

print(f"✓ Assets: {list(item.assets.keys())}")
print(f"✓ Visual asset href: {item.assets['visual'].href}")
assert item.datetime == datetime.datetime(2023, 6, 15, 12, 0, 0)
assert item.properties['cloud_cover'] == 15.5 and item.properties['platform'] == 'satellite-1'
assert list(item.assets) == ['visual', 'metadata']
assert item.assets['visual'].roles == ['visual']
assert item.assets['metadata'].media_type == 'application/json'

# As GeoJSON, an item is a Feature with the geometry and the assets
item_dict = item.to_dict()
assert item_dict['type'] == 'Feature'
assert item_dict['geometry'] == {"type": "Point", "coordinates": [-122.4, 37.8]}
assert item_dict['assets']['visual']['href'] == 'https://example.com/visual.tif'

## Example 3: Serialization and deserialization

In [ ]:
from pystac import Catalog
import json

# Example 3: Serialization and deserialization
print("Testing pystac serialization:")

# Create catalog
catalog = Catalog(id='example-catalog', description='Example catalog')

# Serialize to dictionary
catalog_dict = catalog.to_dict()
print(f"✓ Serialized to dict")
print(f"✓ Dict type: {catalog_dict['type']}")
print(f"✓ Dict keys: {list(catalog_dict.keys())}")

# Serialize to JSON
catalog_json = json.dumps(catalog_dict, indent=2)
print(f"✓ JSON length: {len(catalog_json)} chars")

# Clone catalog
cloned = catalog.clone()
print(f"✓ Cloned catalog: {cloned.id}")
print(f"✓ Same ID: {cloned.id == catalog.id}")

# Full copy
copied = catalog.full_copy()
print(f"✓ Full copy: {copied.id}")
print(f"✓ Is different object: {copied is not catalog}")
assert catalog_dict['type'] == 'Catalog' and catalog_dict['id'] == 'example-catalog'
assert {'type', 'id', 'stac_version', 'description', 'links'} <= set(catalog_dict)
assert json.loads(catalog_json) == catalog_dict

# Reading the dictionary back gives an equivalent catalog
restored = Catalog.from_dict(catalog_dict)
assert restored.id == catalog.id and restored.description == catalog.description
assert cloned.id == catalog.id and cloned is not catalog
assert copied.id == catalog.id and copied is not catalog

## Example 4: Links and relationships

In [ ]:
from pystac import Catalog, Link

# Example 4: Links and relationships
print("Testing pystac Links:")

# Create catalogs
root = Catalog(id='root', description='Root catalog')
child = Catalog(id='child', description='Child catalog')

# Add child relationship
root.add_child(child)
print(f"✓ Added child to root")

# Get links
root_links = list(root.get_links())
print(f"✓ Root links: {len(root_links)}")
for link in root_links:
    print(f"  {link.rel}: {link.target}")

child_links = list(child.get_links())
print(f"✓ Child links: {len(child_links)}")
for link in child_links:
    print(f"  {link.rel}: {link.target}")

# Add custom link
custom_link = Link(rel='related', target='https://example.com')
root.add_link(custom_link)
print(f"✓ Added custom link")
print(f"✓ Total root links: {len(list(root.get_links()))}")
assert sorted(link.rel for link in root_links) == ['child', 'root']
assert sorted(link.rel for link in child_links) == ['parent', 'root']
assert child.get_parent() is root and child.get_root() is root
assert [link.target for link in root.get_links('related')] == ['https://example.com']
assert len(list(root.get_links())) == 3

## Example 5: ItemCollection operations

In [ ]:
from pystac import Item, ItemCollection
import datetime

# Example 5: ItemCollection operations
print("Testing pystac ItemCollection:")

# Create multiple items
items = []
for i in range(3):
    item = Item(
        id=f'item-{i}',
        geometry={"type": "Point", "coordinates": [-122.4 + i*0.1, 37.8]},
        bbox=[-122.4 + i*0.1, 37.8, -122.4 + i*0.1, 37.8],
        datetime=datetime.datetime(2023, 6, 15 + i),
        properties={'cloud_cover': 10.0 + i*5}
    )
    items.append(item)

print(f"✓ Created {len(items)} items")

# Create item collection
ic = ItemCollection(items)
print(f"✓ ItemCollection items: {len(ic.items)}")

# Serialize to dict
ic_dict = ic.to_dict()
print(f"✓ Dict type: {ic_dict['type']}")
print(f"✓ Features: {len(ic_dict['features'])}")

# Filter items
low_cloud = [i for i in ic.items if i.properties['cloud_cover'] < 15]
print(f"✓ Low cloud items: {len(low_cloud)}")
assert [item.id for item in ic.items] == ['item-0', 'item-1', 'item-2']
assert ic_dict['type'] == 'FeatureCollection' and len(ic_dict['features']) == 3
assert [item.id for item in low_cloud] == ['item-0']   # cloud_cover 10, 15, 20

# Round trip through the FeatureCollection dictionary
restored = ItemCollection.from_dict(ic_dict)
assert [item.id for item in restored] == ['item-0', 'item-1', 'item-2']
assert [item.properties['cloud_cover'] for item in restored] == [10.0, 15.0, 20.0]